In [1]:
import pandas as pd
import random
import json
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)

2026-02-14 13:46:24.363514: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771076784.731416      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771076784.838582      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771076785.812763      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771076785.812807      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771076785.812809      55 computation_placer.cc:177] computation placer alr

In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/deepgopal
/kaggle/input/datasets/deepgopal/recipe-dataset


In [3]:
df = pd.read_csv("/kaggle/input/datasets/deepgopal/recipe-dataset/recipes_cleaned_1.csv")

all_ings = set()

for row in df["input"]:
    for ing in row.split(","):
        all_ings.add(ing.strip().lower())

ingredient_list = sorted(list(all_ings))

print("Total ingredients:", len(ingredient_list))

with open("ingredient_vocab.txt", "w") as f:
    for ing in ingredient_list:
        f.write(ing + "\n")

Total ingredients: 3245


In [4]:
templates = [
    "I have {ings}",
    "Can I cook something with {ings}",
    "Suggest a recipe using {ings}",
    "I only have {ings} at home",
    "What can I make with {ings}",
    "I found {ings} in my kitchen, any recipe ideas",
    "Give me a simple dish using {ings}",
    "I want to cook using {ings}, what do you suggest",
    "Are there any good recipes with {ings}",
    "Help me prepare something tasty with {ings}"
]

def generate_ner_dataset(ingredient_list, num_samples=1500):

    dataset = []

    for _ in range(num_samples):
        k = random.randint(1, 4)
        chosen = random.sample(ingredient_list, k)

        sentence = random.choice(templates).format(
            ings=", ".join(chosen)
        )

        dataset.append({
            "text": sentence,
            "ingredients": chosen
        })

    return dataset


data = generate_ner_dataset(ingredient_list)

with open("ner_raw.json", "w") as f:
    json.dump(data, f, indent=2)

In [5]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

label2id = {"O": 0, "B-ING": 1, "I-ING": 2}
id2label = {v: k for k, v in label2id.items()}

MAX_LEN = 64

def tokenize_and_align(example):

    text = example["text"]
    ingredients = example["ingredients"]

    tokenized = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

    labels = [-100] * len(tokenized["input_ids"])

    for ing in ingredients:
        start = text.lower().find(ing)
        if start == -1:
            continue
        end = start + len(ing)

        for i, (s, e) in enumerate(tokenized["offset_mapping"]):

            if s == 0 and e == 0:
                continue

            if s >= start and e <= end:
                if s == start:
                    labels[i] = label2id["B-ING"]
                else:
                    labels[i] = label2id["I-ING"]
            elif labels[i] == -100:
                labels[i] = label2id["O"]

    tokenized["labels"] = labels
    tokenized.pop("offset_mapping")

    return tokenized


dataset = Dataset.from_list(data)
dataset = dataset.map(tokenize_and_align)

dataset.set_format("torch")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./ingredient_extractor",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    num_train_epochs=10,
    logging_steps=50,
    save_strategy="epoch",

    save_total_limit=2,      
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

trainer.train()


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
50,0.117800
100,0.005000
150,0.002300
200,0.003700
250,0.002700
300,0.001700
350,0.000600
400,0.000900
450,0.000400


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

TrainOutput(global_step=470, training_loss=0.014375487720950486, metrics={'train_runtime': 83.0745, 'train_samples_per_second': 180.561, 'train_steps_per_second': 5.658, 'total_flos': 244978992000000.0, 'train_loss': 0.014375487720950486, 'epoch': 10.0})

In [9]:
trainer.save_model("./ingredient_extractor")
tokenizer.save_pretrained("./ingredient_extractor")

print("Final model saved.")

Final model saved.


In [10]:
import os
import shutil

model_dir = "/kaggle/working/ingredient_extractor"

for item in os.listdir(model_dir):
    if item.startswith("checkpoint"):
        shutil.rmtree(os.path.join(model_dir, item))

print("All checkpoints removed.")

All checkpoints removed.


In [12]:
!zip -r ingredient_extractor.zip /kaggle/working/ingredient_extractor

  adding: kaggle/working/ingredient_extractor/ (stored 0%)
  adding: kaggle/working/ingredient_extractor/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 8%)
  adding: kaggle/working/ingredient_extractor/special_tokens_map.json (deflated 42%)
  adding: kaggle/working/ingredient_extractor/tokenizer_config.json (deflated 75%)
  adding: kaggle/working/ingredient_extractor/vocab.txt (deflated 53%)
  adding: kaggle/working/ingredient_extractor/training_args.bin (deflated 54%)
  adding: kaggle/working/ingredient_extractor/config.json (deflated 46%)
  adding: kaggle/working/ingredient_extractor/tokenizer.json (deflated 71%)


In [11]:
from IPython.display import FileLink
FileLink('/kaggle/working/ingredient_extractor.zip')

/kaggle/working/ingredient_extractor.zip